# 00 - Esegui B1, B2 e A1 in fila

Kaggle dà a ogni notebook la sua cartella `/kaggle/working`: se B1, B2 e A1 girano come notebook
separati, A1 non vede gli output degli altri due. Questo notebook li esegue uno dopo l'altro nella
stessa sessione, così ognuno legge gli output del precedente da `dfl_outputs`.

| Passo | Modulo | Produce | Usato da |
|---|---|---|---|
| 1. B1 palla morta | `pipeline/deadball.py` | `<match>_deadball.npz` (`p_dead`) + modello in `models/` | B2, A1 |
| 2. B2 tocco + controllo | `pipeline/touch_control.py --dead auto` | `<match>_rule.npz` (`labels`, `p_home`) | A1 |
| 3b. C1 rete sui giocatori (GPU) | `pipeline/player_net.py` | `<match>_net.npz` (`p_home`) + modello in `models/` | A1 |
| 3. A1 decoder | `pipeline/decoder.py` | `<match>_decoded.npz` + tabella finale | — |

Tempo indicativo con le griglie complete: 30-45 minuti per B1, B2 e A1; con C1 su GPU T4 aggiungere 45-75 minuti. Con `QUICK = True` molto meno.
Tutti i dataset di input devono essere **privati**. Solo partite DFL: nessun dato di BAR-ATH.

## 0. Codice della repo
Se il codice è caricato come dataset Kaggle (qualunque nome), questa cella non fa nulla.
Altrimenti clona il branch con il token GitHub salvato nei Kaggle Secrets come `GITHUB_TOKEN`
(Add-ons → Secrets) e richiede **Internet attivo**.

In [ ]:
import os, glob, subprocess
BRANCH = "feature/notebooks-dfl"
DEST = "/kaggle/working/Hackaton_Barcellona"
code_ds = glob.glob("/kaggle/input/**/pipeline/decoder.py", recursive=True)
if code_ds:
    print("codice trovato come dataset:", os.path.dirname(os.path.dirname(code_ds[0])), "- clone non necessario")
elif os.path.isdir(DEST):
    print("clone gia presente")
elif os.path.isdir("/kaggle/input"):
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = "https://{}@github.com/samugarofalo/Hackaton_Barcellona.git".format(tok)
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, DEST], capture_output=True, text=True)
    print("clone ok" if r.returncode == 0 else r.stderr.replace(tok, "***"))
else:
    print("non su Kaggle: clone non necessario")


## 1. Setup
Stessi input dei singoli notebook: codice della repo, partite DFL convertite (`DFL-MAT-*.npz`) e, consigliato, `ball_runs.npz`. Vedi `notebooks/README.md`.

In [ ]:
import os, sys, subprocess, time, glob
_found = sorted(glob.glob("/kaggle/input/**/pipeline/decoder.py", recursive=True))  # codice caricato come dataset, ovunque sia montato
REPO = os.environ.get("MTAH_REPO") or next((p for p in ["/kaggle/input/hackaton-barcellona-code", "/kaggle/working/Hackaton_Barcellona", os.path.abspath("..")] + [os.path.dirname(os.path.dirname(f)) for f in _found] if os.path.isdir(os.path.join(p, "pipeline"))), None)
assert REPO, "Imposta MTAH_REPO alla cartella della repo"
sys.path[:0] = [REPO, os.path.join(REPO, "pipeline"), os.path.join(REPO, "starter"), os.path.join(REPO, "evaluation")]
print("repo:", REPO)

QUICK = False   # True = griglie ridotte, per una prova veloce
RUN_C1 = True   # rete sui giocatori: serve Accelerator = GPU T4 (su CPU e molto lenta)

# Partite DFL: se ci sono solo gli XML originali, convertili una volta in /kaggle/working
if not os.environ.get("DFL_CONVERTED") and not glob.glob("/kaggle/input/**/DFL-MAT-*.npz", recursive=True):
    xml = sorted(glob.glob("/kaggle/input/**/DFL_04_03_positions_raw_observed_*.xml", recursive=True))
    if xml:
        os.environ["DFL_SRC"] = os.path.dirname(xml[0])
        os.environ["DFL_CONVERTED"] = "/kaggle/working/dfl_converted"
        print("converto", len(xml), "partite da", os.environ["DFL_SRC"], "(qualche minuto)...", flush=True)
        r = subprocess.run([sys.executable, os.path.join(REPO, "pipeline", "convert_dfl.py")], cwd=REPO, env=os.environ.copy(), capture_output=True, text=True)
        print(r.stdout[-3000:], r.stderr[-3000:])

import dfl_paths
print("partite DFL:", len(dfl_paths.match_paths()), "in", dfl_paths.converted_dir())
print("output in:", dfl_paths.out_dir())
runs = sorted(glob.glob("/kaggle/input/**/ball_runs.npz", recursive=True))
print("ball_runs.npz:", runs[0] if runs else "NON trovato: si useranno durate indicative (placeholder)")
# i sottoprocessi devono scrivere e leggere nelle stesse cartelle
os.environ["DFL_OUT"] = dfl_paths.out_dir()
os.environ["DFL_CONVERTED"] = dfl_paths.converted_dir()
if runs: os.environ.setdefault("DFL_BALL_RUNS", runs[0])
os.environ.setdefault("PLAYER_NET_MODELS", "/kaggle/working/models" if dfl_paths.on_kaggle() else os.path.join(REPO, "models"))
os.environ.setdefault("DEADBALL_MODELS", "/kaggle/working/models" if dfl_paths.on_kaggle() else os.path.join(REPO, "models"))

In [ ]:
def run(script, *args):
    cmd = [sys.executable, os.path.join(REPO, "pipeline", script), *args]
    print("$", " ".join(cmd), flush=True)
    t = time.time()
    p = subprocess.run(cmd, cwd=REPO, env=os.environ.copy(), capture_output=True, text=True)
    print(p.stdout[-20000:])
    if p.returncode != 0:
        print(p.stderr[-5000:])
        raise RuntimeError("{} fallito (codice {})".format(script, p.returncode))
    print("--- {} finito in {:.1f} min".format(script, (time.time() - t) / 60))
    print()


## 2. B1: rilevatore di palla morta
Addestra LightGBM lasciando fuori una partita alla volta e scrive `p_dead` per ogni partita. Da guardare: spellF1 a valle con la `D` del rilevatore contro la `D` = palla mancante.

In [ ]:
run("deadball.py", *(["--quick"] if QUICK else []))

## 3. B2: tocco + conferma del controllo
`--dead auto` usa la palla morta di B1 appena scritta. Da guardare: spellF1 di tocco + controllo contro il metodo 4 e il baseline, e i possessi predetti contro veri.

In [ ]:
run("touch_control.py", "--dead", "auto", *(["--quick"] if QUICK else []))

## 3b. C1: rete sui giocatori (GPU)
Stima chi ha la palla dai 22 giocatori, anche quando la palla manca, e scrive `<match>_net.npz`, che A1
legge da solo. Griglia: finestra 4 o 10 s, 32/64/128 canali, scelta lasciando fuori una partita alla
volta. Con la GPU T4 circa 45-75 minuti (con `QUICK` pochi minuti). Se `RUN_C1 = False` il passo viene
saltato e A1 lavora senza la rete. Da guardare: accuratezza nei frame con la palla nel vuoto, rete
contro giocatore più vicino.

In [ ]:
if RUN_C1:
    import torch
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    print("device:", dev, "(attiva Accelerator = GPU T4 se dice cpu)")
    run("player_net.py", "--device", dev, *(["--quick"] if QUICK else []))
else:
    print("C1 saltato")

## 4. A1: decoder a segmenti
Legge da solo `p_dead` (B1) e `p_home` (B2). Nella riga `sources` deve comparire `p_dead: B1 file` `p_home B2: yes` e, se C1 ha girato, `p_home C1 net: yes`: se no, gli output dei passi precedenti non sono stati trovati.

In [ ]:
import importlib, decoder
importlib.reload(decoder)
res = decoder.evaluate_dfl()
res["summary"]

## 5. Cosa riportare al team
Solo numeri aggregati su DFL:

- tabella finale di A1 (spellF1 medio, min, max per metodo);
- riga `sources` (quali output sono stati usati, `ball_runs` misurato o placeholder);
- dai passi 2 e 3: spellF1 della palla morta di B1 e di tocco + controllo.

Differenze sotto circa 0,04 di spellF1 sono rumore.

In [ ]:
print("sources:", res["sources"])
print("parametri scelti nei fold:", sorted(set(res["params"].values())))
print("file prodotti:", sorted(os.listdir(dfl_paths.out_dir())))